In [5]:
from langchain.chat_models import init_chat_model
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028AA3E25A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028AA3F1A790>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(description="This is the title of the movie")
    year: int = Field(description="This is the year the movie was released")
    rating: float = Field(description="This is the rating of the movie")
    director: str = Field(description="this is the director of the movie")

In [7]:
model.invoke("tell me about the movie Inception")


AIMessage(content='**Inception** (2009) – A mind‑bending thriller directed by Christopher\u202fNolan\n\n| Category | Details |\n|----------|---------|\n| **Genre** | Science‑fiction, action, psychological thriller |\n| **Runtime** | 148 minutes |\n| **Release** | 2010 (United States) |\n| **Budget** | ~\u202f160\u202fmillion\u202fUSD |\n| **Box‑office** | Over\u202f830\u202fmillion\u202fUSD worldwide |\n| **Director** | Christopher\u202fNolan |\n| **Screenwriter** | Christopher\u202fNolan |\n| **Producer** | Emma Thomas, Christopher\u202fNolan |\n| **Music** | Hans\u202fZimmer |\n| **Cinematography** | Wally\u202fPfister |\n| **Main cast** | • Leonardo\u202fDiCaprio as Dom\u202fCobb<br>• Joseph\u202fGordon‑Levitt as Arthur<br>• Ellen\u202fPage as Ariadne<br>• Tom\u202fHardy as Eames<br>• Ken\u202fWatanabe as Yusuf<br>• Cillian\u202fMurphy as Saito<br>• Michael\u202fCaine as Professor\u202fSaito (cameo) |\n\n---\n\n### Premise\n\nA skilled “extractor” (Dom\u202fCobb) specializes in stea

In [13]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure
response = model_with_structure.invoke("tell me about the movie Interstellar")
response



Movie(title='Interstellar', year=2014, rating=8.6, director='Christopher Nolan')

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(...,description="This is the title of the movie")
    year: int = Field(...,description="This is the year the movie was released")
    rating: float = Field(...,description="This is the rating of the movie")
    director: str = Field(...,description="this is the director of the movie")

In [21]:
model_with_structure = model.with_structured_output(Movie,include_raw=True)
model_with_structure
response = model_with_structure.invoke("tell me about the movie Interstellar")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants info about the movie Interstellar. We can provide summary, cast, director, rating, year. We can call function to get data? The function expects director, rating, title, year. But we need to supply those values. We can call function with the movie's data: director Christopher Nolan, rating maybe 8.6 (IMDb?), title Interstellar, year 2014. The function returns any. So we call it. Then we can give answer. Probably they want details. We'll call the function.", 'tool_calls': [{'id': 'fc_c5914773-0b2a-4900-a957-7011b1007cb9', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.6,"title":"Interstellar","year":2014}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 164, 'total_tokens': 311, 'completion_time': 0.189176681, 'completion_tokens_details': {'reasoning_tokens': 108}, 'prompt_time': 0.008103724, 'prompt_tokens_detail

In [ ]:
### nested Structure

from pydantic import BaseModel,Field

class Actor(BaseModel):
    name: str
    role: str
    
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float = Field(description="Budget of the movie in USD") 
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("tell me about the movie The Dark Knight")
response

MovieDetails(title='The Dark Knight', year=2008, cast=[Actor(name='Christian Bale', role='Bruce Wayne / Batman'), Actor(name='Heath Ledger', role='Joker'), Actor(name='Aaron Eckhart', role='Harvey Dent / Two-Face'), Actor(name='Michael Caine', role='Alfred')], genre=['Action', 'Crime', 'Drama'], budget=185000000.0)

In [25]:
### Typed Dict

from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"The year the movie was released"]
    genre: Annotated[list[str],...,"The genre of the movie"]
    director: Annotated[str,...,"The director of the movie"]
model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("tell me about the movie The Matrix")
response

{'director': 'The Wachowskis',
 'genre': ['Action', 'Sci-Fi'],
 'title': 'The Matrix',
 'year': 1999}

In [ ]:
class Actor(TypedDict):
    name: str
    role: str
    
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float= Field(description="Budget of the movie in USD") 
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("tell me about the movie The Dark Knight")
response

{'budget': 185000000,
 'cast': [{'name': 'Christian Bale', 'role': 'Bruce Wayne / Batman'},
  {'name': 'Heath Ledger', 'role': 'Joker'},
  {'name': 'Aaron Eckhart', 'role': 'Harvey Dent / Two-Face'},
  {'name': 'Michael Caine', 'role': 'Alfred Pennyworth'},
  {'name': 'Gary Oldman', 'role': 'Commissioner James Gordon'}],
 'genre': ['Action', 'Crime', 'Drama'],
 'title': 'The Dark Knight',
 'year': 2008}

In [27]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [14]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent


class Contactinfo(BaseModel):
    name: str = Field(...,description="Name of the person")
    email: str = Field(...,description="Email of the person")
    phone: str = Field(...,description="phone number of the person")



agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format = Contactinfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact information from John Doe, doe@example.com, 123-456-7890"
        }
    ]
})
result["structured_response"]
    


Contactinfo(name='John Doe', email='doe@example.com', phone='123-456-7890')

In [16]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class Contactinfo(BaseModel):
    name: str 
    email: str
    phone: str


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format = Contactinfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact information from John Doe, doe@example.com, 123-456-7890"
        }
    ]
})
result["structured_response"]
    


Contactinfo(name='John Doe', email='doe@example.com', phone='123-456-7890')

In [17]:
from dataclasses import dataclass
from langchain.agents import create_agent


class Contactinfo(BaseModel):
    name: str 
    email: str
    phone: str


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format = Contactinfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact information from John Doe, doe@example.com, 123-456-7890"
        }
    ]
})
result["structured_response"]
    


Contactinfo(name='John Doe', email='doe@example.com', phone='123-456-7890')

In [3]:
### Summarization Middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

model = create_agent(
    model = "groq:openai/gpt-oss-20b",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:openai/gpt-oss-20b",
            trigger = ("messages",10),
            keep = ("messages",4)
            
        )
    ]
    
)



In [21]:
### thread id
config={"configurable":{"thread_id":"test-2"}}

In [22]:
questions = [
    "what is 2-2",
    "what is 2+2",
    "what is 4*7",
    "what is 4/2",
    "what is 4-2",
    "what is 4+2",
]

for q in questions:
    response = model.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(response)
    print(f"length of messages {len(response['messages'])}")
    

{'messages': [HumanMessage(content='what is 2-2', additional_kwargs={}, response_metadata={}, id='a43030f3-5437-4882-8e5d-e5e2890828bf'), AIMessage(content='2\u202f–\u202f2\u202f=\u202f0.', additional_kwargs={'reasoning_content': 'The user asks: "what is 2-2". Likely they want the result of 2 minus 2, which is 0. The user might be testing. Provide simple answer: 0. Could also mention that it\'s subtraction. So answer: 0.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 77, 'total_tokens': 153, 'completion_time': 0.081492395, 'completion_tokens_details': {'reasoning_tokens': 57}, 'prompt_time': 0.005197393, 'prompt_tokens_details': None, 'queue_time': 0.048849013, 'total_time': 0.086689788}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0c9ca-3cee-7220-9988-b35cb65fcd56-0', tool_calls=[], invalid_tool_calls=[], usag

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

def read_email_tool(email_id : str) -> str:
    """Mock function to read a email given an email id"""
    return f"email content for email id {email_id}"
def send_email_tool(recipient : str, subject : str, body : str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject {subject} and body {body}"


In [29]:
agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [ HumanInTheLoopMiddleware(
                    interrupt_on = {
                        "send_email_tool": {
                            "allowed_decisions": ["approve","reject","edit"]
                        },
                        "read_email_tool":False,
                    }
                   
        )
    ]
)
                    

In [26]:
from langchain_core.messages import SystemMessage, HumanMessage
config={"configurable":{"thread_id":"test-approve"}}

result = agent.invoke({
    "messages": [HumanMessage(content="Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John")]
}, config=config)
result

{'messages': [HumanMessage(content='Send email to john@example.com with subject Hello and body Hi John', additional_kwargs={}, response_metadata={}, id='25a9d1c3-513c-494a-b5ef-0b966eb4835a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email via tool send_email_tool.', 'tool_calls': [{'id': 'fc_30c106b4-2dd9-4e27-9a93-b7d65bf22fb0', 'function': {'arguments': '{"body":"Hi John","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 170, 'total_tokens': 217, 'completion_time': 0.079193348, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.01167745, 'prompt_tokens_details': None, 'queue_time': 0.211125468, 'total_time': 0.090870798}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq

In [ ]:
### approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("paused approving email!...")
    
    result = agent.invoke(
        Command(
            resume={"decisions":[
                {"type":"approve"}
                ]
            }
            
        ),
        config=config
    )
    print("final result",result["messages"][-1].content)

paused approving email!...
final result ✅ Email sent to ragulpradhive10@gmail.com with subject "Hello" and body "Hi John".


In [28]:
from langchain_core.messages import SystemMessage, HumanMessage
config={"configurable":{"thread_id":"test-reject"}}

result = agent.invoke({
    "messages": [HumanMessage(content="Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John")]
}, config=config)
result

{'messages': [HumanMessage(content='Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John', additional_kwargs={}, response_metadata={}, id='902c5adb-4aa5-41d3-81cc-7512411f2cad'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_d3e90bbe-ab1e-479f-b97b-e1cad47bf1ee', 'function': {'arguments': '{"body":"Hi John","recipient":"ragulpradhive10@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 175, 'total_tokens': 224, 'completion_time': 0.050329032, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008529285, 'prompt_tokens_details': None, 'queue_time': 0.346336773, 'total_time': 0.058858317}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'g

In [29]:
### reject 

from langgraph.types import Command

if "__interrupt__" in result:
    print("paused approving email!...")
    
    result = agent.invoke(
        Command(
            resume={"decisions":[
                {"type":"reject"}
                ]
            }
            
        ),
        config=config
    )
    print("final result",result["messages"][-1].content)

paused approving email!...
final result I’m sorry, but I couldn’t send the email at this time.  
If you’d like me to try again or if you need anything else, just let me know!


In [30]:
result

{'messages': [HumanMessage(content='Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John', additional_kwargs={}, response_metadata={}, id='902c5adb-4aa5-41d3-81cc-7512411f2cad'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_d3e90bbe-ab1e-479f-b97b-e1cad47bf1ee', 'function': {'arguments': '{"body":"Hi John","recipient":"ragulpradhive10@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 175, 'total_tokens': 224, 'completion_time': 0.050329032, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008529285, 'prompt_tokens_details': None, 'queue_time': 0.346336773, 'total_time': 0.058858317}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'g

In [40]:
from langchain_core.messages import SystemMessage, HumanMessage
config={"configurable":{"thread_id":"test-edit3"}}

result = agent.invoke({
    "messages": [HumanMessage(content="Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John")]
}, config=config)
result

{'messages': [HumanMessage(content='Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John', additional_kwargs={}, response_metadata={}, id='57559a61-5e9d-41ab-96af-eb351a117df3'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. We have a tool to send email. We need to call send_email_tool with body, recipient, subject. The body: "Hi John", subject: "Hello", recipient: "ragulpradhive10@gmail.com". Let\'s call the function.', 'tool_calls': [{'id': 'fc_8251ea97-649b-4eab-80c7-53beef21f00b', 'function': {'arguments': '{"body":"Hi John","recipient":"ragulpradhive10@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 175, 'total_tokens': 275, 'completion_time': 0.111262267, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.009979668, 'prompt_tokens_details': None, 'queue_time': 0.332990057, '

In [41]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused: editing email!...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    for msg in result["messages"]:

        if type(msg).__name__ == "ToolMessage":
            print("Tool result:", msg.content)

    print("Email workflow completed.")

Paused: editing email!...
Tool result: Email sent to correct@email.com with subject Corrected Subject and body This was edited by human before sending
Email workflow completed.


In [ ]:
result

{'messages': [HumanMessage(content='Send email to ragulpradhive10@gmail.com with subject Hello and body Hi John', additional_kwargs={}, response_metadata={}, id='fb2a862f-8757-4419-92d9-bb847dae03c0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the send_email_tool.', 'tool_calls': [{'id': 'fc_2bc4479a-657f-4090-b1d9-0472df33289b', 'function': {'arguments': '{"body":"Hi John","recipient":"ragulpradhive10@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 175, 'total_tokens': 225, 'completion_time': 0.051996501, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.009886213, 'prompt_tokens_details': None, 'queue_time': 0.40089183, 'total_time': 0.061882714}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider'

: 